In [0]:
from pyspark.sql import functions as F

catalog = "dbw_company_risk"
companies = spark.table(f"{catalog}.silver.companies")

status_points = (
    F.when(F.col("status_group").isin("Liquidation", "Administration or receivership"), 80)
     .when(F.col("status_group") == "Proposal to strike off", 50)
     .when(F.col("status_group") == "Voluntary arrangement", 40)
     .otherwise(0)
)
age = F.col("company_age_years")
age_points = F.when(age < 1, 15).when(age < 3, 8).otherwise(0)

scored = (
    companies
    .withColumn("pts_status", status_points)
    .withColumn("pts_accounts", F.when(F.col("accounts_overdue"), 25).otherwise(0))
    .withColumn("pts_confirmation", F.when(F.col("confirmation_overdue"), 15).otherwise(0))
    .withColumn("pts_dormant", F.when(F.upper("accounts_category") == "DORMANT", 10).otherwise(0))
    .withColumn("pts_age", F.when(
         (F.col("pts_status") + F.col("pts_accounts")
          + F.col("pts_confirmation") + F.col("pts_dormant")) > 0,
         age_points).otherwise(0))
    .withColumn("risk_score", F.least(
        F.lit(100),
        F.col("pts_status") + F.col("pts_accounts") + F.col("pts_confirmation")
        + F.col("pts_age") + F.col("pts_dormant")))
    .withColumn("risk_band",
        F.when(F.col("risk_score") >= 50, "High")
         .when(F.col("risk_score") >= 20, "Medium")
         .otherwise("Low"))
    .withColumn("risk_reasons", F.array_compact(F.array(
        F.when(F.col("pts_status") > 0, F.concat(F.lit("Status: "), F.col("status_group"))),
        F.when(F.col("accounts_overdue"), F.lit("Accounts overdue")),
        F.when(F.col("confirmation_overdue"), F.lit("Confirmation statement overdue")),
        F.when(F.col("pts_age") > 0, F.lit("Young company with other warnings")),
        F.when(F.col("pts_dormant") > 0, F.lit("Filed as dormant")),
    )))
)

(scored.select(
        "company_number", "company_name", "status_group", "sic_code", "sic_description",
        "postcode_area", "company_age_years", "accounts_overdue", "confirmation_overdue",
        "risk_score", "risk_band", "risk_reasons", "snapshot_date")
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.company_risk"))

print("gold.company_risk created")

In [0]:
by_industry = (
    spark.table(f"{catalog}.gold.company_risk")
    .where("status_group = 'Active' AND sic_description IS NOT NULL")
    .groupBy("sic_code", "sic_description")
    .agg(
        F.count("*").alias("companies"),
        F.round(F.avg("risk_score"), 1).alias("avg_risk_score"),
        F.round(100 * F.avg(F.when(F.col("risk_band") != "Low", 1).otherwise(0)), 1)
            .alias("pct_medium_or_high"))
    .where("companies >= 1000")
)

(by_industry.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.risk_by_industry"))

display(by_industry.orderBy(F.desc("pct_medium_or_high")).limit(15))

In [0]:
%sql
SELECT company_number, company_name, status_group, risk_score, risk_band, risk_reasons
FROM dbw_company_risk.gold.company_risk
WHERE company_number IN ('00445790', '00102498', '01833679')

In [0]:
%sql
SELECT reason, COUNT(*) AS active_companies
FROM (
    SELECT explode(risk_reasons) AS reason
    FROM dbw_company_risk.gold.company_risk
    WHERE status_group = 'Active'
)
GROUP BY reason
ORDER BY active_companies DESC